<a href="https://colab.research.google.com/github/Dr-Isam-ALJAWARNEH/fds-project-airnav/blob/main/Code/AirQuality_Clustering_Regression_Map_NYC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EDA

Correlation

Clustering & Regression

Time/sample vs. RMSE

Geohas:
-Spatial grouping

-Clustering by location

-Heatmaps, pivot tables, etc.

In [ ]:
# Step 1: Install all required libraries
!pip install datascience h3 pandas matplotlib seaborn scikit-learn

In [ ]:
# Step 2: Upload the cleaned CSV file
from google.colab import files
uploaded = files.upload()

In [ ]:
# Step 3: Load the uploaded CSV into a pandas DataFrame and convert it to a datascience Table
import pandas as pd
from datascience import Table

# Extract the uploaded file name
csv_file = list(uploaded.keys())[0]

# Read the CSV using pandas
df = pd.read_csv(csv_file)

# Convert pandas DataFrame to datascience Table for easier display/preview
table = Table.from_df(df)

# Display the first 5 rows in table format
table.show(5)

# Expected output: A nicely formatted interactive table with 5 rows and all columns from your dataset,
# such as: time, latitude, longitude, bin0–bin23, temperature, humidity, pm25, etc.

In [ ]:
# Step 4: Display summary statistics for all numeric columns
# This includes count, mean, std deviation, min, 25%, 50%, 75%, and max values.
print("Descriptive Statistics:")
display(df.describe())

# Expected output:
# A table showing statistical summaries (mean, std, min, max, etc.)
# for columns like latitude, longitude, bin0–bin23, temperature, humidity, pm25, pm1, pm10, and Cluster.

In [ ]:
# Step 4: Display summary statistics for all numeric columns
# This includes count, mean, std deviation, min, 25%, 50%, 75%, and max values.
print("Descriptive Statistics:")
display(df.describe())

# Expected output:
# A table showing statistical summaries (mean, std, min, max, etc.)
# for columns like latitude, longitude, bin0–bin23, temperature, humidity, pm25, pm1, pm10, and Cluster.

In [ ]:
# Step 5: Plot histograms for each numeric column in the dataset

# Ensure inline plotting in Google Colab
%matplotlib inline

import matplotlib.pyplot as plt

# Select only numeric columns
numeric_df = df.select_dtypes(include='number')

print("Histograms for All Numeric Columns:")

# Check and plot
if not numeric_df.empty:
    for col in numeric_df.columns:
        print(f"Plotting histogram for: {col}")
        plt.figure(figsize=(8, 4))
        plt.hist(numeric_df[col].dropna(), bins=30, color='skyblue', edgecolor='black')
        plt.title(f"Histogram of {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        plt.close()
else:
    print("No numeric columns found in the dataset.")

# Expected output:
# A sequence of individual histograms will be displayed — one for each numeric column such as:
# 'temperature', 'humidity', 'pm25', 'bin0', ..., 'bin23', 'Cluster', etc.
# Each histogram shows the frequency distribution of values in that column.

In [ ]:
# Step 6: Plot enhanced boxplots (excluding time and bin0–bin23, zoomed y-axis)

%matplotlib inline

import seaborn as sns
import matplotlib.pyplot as plt

# Step 6.1: Select numeric columns
numeric_df = df.select_dtypes(include='number')

# Step 6.2: Exclude time and bin0–bin23 columns
columns_to_exclude = [f'bin{i}' for i in range(24)] + ['time']
columns_to_keep = [col for col in numeric_df.columns if col not in columns_to_exclude]

filtered_df = numeric_df[columns_to_keep]

print("Enhanced Boxplots for Numeric Columns (excluding time and bins):")

# Step 6.3: Plot with y-axis zoom (ignoring extreme outliers above 1000)
if not filtered_df.empty:
    plt.figure(figsize=(18, 6))
    sns.boxplot(data=filtered_df)
    plt.title("Boxplots (Excluding time and bin0–bin23) – Zoomed In")
    plt.ylim(0, 1000)  # Adjust this if needed
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns available for plotting.")

In [ ]:
# Step 7: Plot time-series data if 'timestamp' column exists in the dataset

# Ensure inline plotting is active
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd

# Check if timestamp column is present
if 'timestamp' in df.columns:
    print("Generating Time-Series Plot...")

    # Convert to datetime format if not already
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    # Drop rows with invalid timestamps
    df = df.dropna(subset=['timestamp'])

    # Set timestamp as index
    df.set_index('timestamp', inplace=True)

    # Plot all numeric columns over time
    df.select_dtypes(include='number').plot(figsize=(18, 6), title="Time-Series Plot")
    plt.xlabel("Timestamp")
    plt.ylabel("Values")
    plt.tight_layout()
    plt.show()

else:
    print("No 'timestamp' column found. Skipping time-series plot.")

# Expected output:
# A multi-line time-series graph with the x-axis as timestamps and y-axis showing values
# for numeric columns like temperature, pm25, humidity, etc.
# If the column doesn't exist, it prints a skip message.

In [ ]:
# Step 8: Generate a correlation heatmap between all numeric columns

# Ensure inline plotting is active
%matplotlib inline

import seaborn as sns
import matplotlib.pyplot as plt

# Select numeric columns only
numeric_df = df.select_dtypes(include='number')

print("Correlation Matrix Heatmap:")

# Check if we have numeric columns to correlate
if not numeric_df.empty:
    plt.figure(figsize=(14, 10))
    correlation_matrix = numeric_df.corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
    plt.title("Correlation Heatmap of Numeric Columns")
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns found to compute correlation.")

# Expected output:
# A heatmap showing the Pearson correlation coefficients between all pairs of numeric columns.
# Helps identify which variables are highly related (positive or negative).

In [ ]:
# Step 9: K-Means clustering and simple linear regression (fixed RMSE calculation)

# Ensure inline plotting is active
%matplotlib inline

from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# Select numeric columns and drop missing values
numeric_df = df.select_dtypes(include='number').dropna()

print("K-Means Clustering and Linear Regression Modeling:")

# Proceed if at least 2 numeric columns exist
if numeric_df.shape[1] >= 2:
    # K-Means clustering
    kmeans = KMeans(n_clusters=3, random_state=42)
    df['Cluster'] = kmeans.fit_predict(numeric_df)

    # Simple linear regression between first two numeric features
    x_col = numeric_df.columns[0]
    y_col = numeric_df.columns[1]

    model = LinearRegression()
    model.fit(numeric_df[[x_col]], numeric_df[y_col])
    y_pred = model.predict(numeric_df[[x_col]])

    # Calculate RMSE manually (for compatibility)
    mse = mean_squared_error(numeric_df[y_col], y_pred)
    rmse = np.sqrt(mse)

    # Plot the regression result
    plt.figure(figsize=(8, 5))
    plt.scatter(numeric_df[x_col], numeric_df[y_col], alpha=0.5, label='Actual')
    plt.plot(numeric_df[x_col], y_pred, color='red', label='Regression Line')
    plt.title(f"{x_col} vs {y_col} (Linear Regression)\nRMSE = {rmse:.2f}")
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for regression or clustering.")

In [ ]:
# Step 10: RMSE vs. Sample Size Plot using simple linear regression

# Ensure inline plotting is active
%matplotlib inline

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# Select numeric columns and remove missing values
numeric_df = df.select_dtypes(include='number').dropna()

print("RMSE vs. Sample Size Plot:")

# Proceed only if dataset is large enough
if numeric_df.shape[0] >= 100 and numeric_df.shape[1] >= 2:
    x_col = numeric_df.columns[0]
    y_col = numeric_df.columns[1]

    sample_sizes = np.linspace(100, len(numeric_df), 10, dtype=int)
    rmses = []

    for size in sample_sizes:
        subset = numeric_df.iloc[:size]
        model = LinearRegression()
        model.fit(subset[[x_col]], subset[y_col])
        prediction = model.predict(subset[[x_col]])
        mse = mean_squared_error(subset[y_col], prediction)
        rmse = np.sqrt(mse)
        rmses.append(rmse)

    # Plot RMSE vs. Sample Size
    plt.figure(figsize=(10, 5))
    plt.plot(sample_sizes, rmses, marker='o')
    plt.title("RMSE vs. Sample Size")
    plt.xlabel("Sample Size")
    plt.ylabel("Root Mean Squared Error (RMSE)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data or numeric columns to plot RMSE vs Sample Size.")

In [ ]:
# Step 11: Encode latitude/longitude into geohash codes at different precisions

# Ensure inline plotting
%matplotlib inline

# Install geohash2 package (compatible and stable)
!pip install geohash2

import geohash2

print("Geohash Spatial Encoding:")

# Check for lat/lon columns
if {'latitude', 'longitude'}.issubset(df.columns):
    # Drop rows with missing coordinates
    df = df.dropna(subset=['latitude', 'longitude'])

    # Encode into geohash at precision levels 5 (coarse) and 7 (finer)
    df['geohash_5'] = df.apply(lambda row: geohash2.encode(row['latitude'], row['longitude'], precision=5), axis=1)
    df['geohash_7'] = df.apply(lambda row: geohash2.encode(row['latitude'], row['longitude'], precision=7), axis=1)

    # Preview the output
    print("Sample rows with geohash codes:")
    display(df[['latitude', 'longitude', 'geohash_5', 'geohash_7']].head())
else:
    print("Missing 'latitude' or 'longitude' — cannot generate geohash.")

In [ ]:
# Part A: Group data by geohash and compute average air quality and weather

# Group by geohash_5 (coarser resolution) and compute means
agg_df = df.groupby('geohash_5')[['pm25', 'pm10', 'pm1', 'temperature', 'humidity']].mean().reset_index()

# Show a few rows of the aggregated result
print("Average measurements per geohash_5 region:")
display(agg_df.head())

# Expected output:
# A table showing average pm25, pm10, pm1, temperature, and humidity per geohash_5 region.

In [ ]:
# Step: Fix geohash decoding and show average air quality on map

import folium
import geohash2

# Step 1: Re-decode geohash to get clean lat/lon (as float tuples)
decoded_coords = agg_df['geohash_5'].apply(geohash2.decode)

# Step 2: Assign lat/lon correctly (not as strings)
agg_df['lat'] = decoded_coords.apply(lambda x: float(x[0]))
agg_df['lon'] = decoded_coords.apply(lambda x: float(x[1]))

# Step 3: Create map centered on average location
map_center = [agg_df['lat'].mean(), agg_df['lon'].mean()]
m = folium.Map(location=map_center, zoom_start=8)

# Step 4: Add PM2.5 + temperature markers
for _, row in agg_df.iterrows():
    popup_text = f"""
    Geohash: {row['geohash_5']}<br>
    PM2.5: {row['pm25']:.2f}<br>
    Temp: {row['temperature']:.1f}°C
    """
    folium.CircleMarker(
        location=(row['lat'], row['lon']),
        radius=6,
        fill=True,
        fill_opacity=0.7,
        popup=popup_text,
        color='blue'
    ).add_to(m)

# Show the map
m

In [ ]:
colormap = plt.colormaps['YlOrRd']

In [ ]:
# Step: Fix geohash decoding to lat/lon and visualize on map

import folium
import geohash2
import matplotlib.pyplot as plt
import matplotlib.colors as colors

# Ensure 'lat' and 'lon' are correctly decoded from geohash_5
decoded_coords = agg_df['geohash_5'].apply(geohash2.decode)
agg_df['lat'] = decoded_coords.apply(lambda x: float(x[0]))
agg_df['lon'] = decoded_coords.apply(lambda x: float(x[1]))

# Normalize PM2.5 for color mapping
norm = colors.Normalize(vmin=agg_df['pm25'].min(), vmax=agg_df['pm25'].max())
colormap = plt.colormaps['YlOrRd']

# Convert PM2.5 to color hex
def get_color(value):
    rgba = colormap(norm(value))
    return colors.to_hex(rgba)

# Create the folium map
map_center = [agg_df['lat'].mean(), agg_df['lon'].mean()]
m = folium.Map(location=map_center, zoom_start=8)

# Add markers to map
for _, row in agg_df.iterrows():
    folium.CircleMarker(
        location=(row['lat'], row['lon']),
        radius=7,
        fill=True,
        fill_opacity=0.8,
        color=get_color(row['pm25']),
        fill_color=get_color(row['pm25']),
        popup=f"""
        Geohash: {row['geohash_5']}<br>
        PM2.5: {row['pm25']:.2f}<br>
        Temp: {row['temperature']:.1f}°C<br>
        Humidity: {row['humidity']:.1f}%
        """
    ).add_to(m)

# Display the map
m


In [ ]:
import folium
import geohash2
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from branca.element import MacroElement
from jinja2 import Template

# Step 1: Decode geohash_5 into lat/lon (as float)
decoded_coords = agg_df['geohash_5'].apply(geohash2.decode)
agg_df['lat'] = decoded_coords.apply(lambda x: float(x[0]))
agg_df['lon'] = decoded_coords.apply(lambda x: float(x[1]))

# Step 2: Normalize PM2.5 values and choose colormap
norm = colors.Normalize(vmin=agg_df['pm25'].min(), vmax=agg_df['pm25'].max())
colormap = plt.colormaps['YlOrRd']

def get_color(value):
    rgba = colormap(norm(value))
    return colors.to_hex(rgba)

# Step 3: Create the folium map
map_center = [agg_df['lat'].mean(), agg_df['lon'].mean()]
m = folium.Map(location=map_center, zoom_start=8)

# Step 4: Add circle markers with color-coded PM2.5
for _, row in agg_df.iterrows():
    folium.CircleMarker(
        location=(row['lat'], row['lon']),
        radius=7,
        fill=True,
        fill_opacity=0.8,
        color=get_color(row['pm25']),
        fill_color=get_color(row['pm25']),
        popup=f"""
        <b>Geohash:</b> {row['geohash_5']}<br>
        <b>PM2.5:</b> {row['pm25']:.2f}<br>
        <b>Temp:</b> {row['temperature']:.1f}°C<br>
        <b>Humidity:</b> {row['humidity']:.1f}%
        """
    ).add_to(m)

# Step 5: Add custom legend (color scale bar)
legend_html = """
{% macro html(this, kwargs) %}
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    width: 170px;
    height: 180px;
    z-index:9999;
    font-size:14px;
    background-color: white;
    padding: 10px;
    border: 2px solid gray;
    border-radius: 5px;">
    <b>PM2.5 Scale</b><br>
    <i style="background:#ffffb2;width:18px;height:18px;display:inline-block"></i> Low<br>
    <i style="background:#fecc5c;width:18px;height:18px;display:inline-block"></i> Moderate<br>
    <i style="background:#fd8d3c;width:18px;height:18px;display:inline-block"></i> High<br>
    <i style="background:#f03b20;width:18px;height:18px;display:inline-block"></i> Very High<br>
    <i style="background:#bd0026;width:18px;height:18px;display:inline-block"></i> Hazardous
</div>
{% endmacro %}
"""

class Legend(MacroElement):
    def __init__(self):
        super().__init__()
        self._template = Template(legend_html)

# Step 6: Add legend to the map
m.get_root().add_child(Legend())

# Step 7: Display map
m

Refine Clustering and Regression Models
(adjusted step number as per your full workflow)

This step includes:

- Spatial-environmental clustering

- Multiple regression to predict PM2.5

- Evaluation using RMSE and R²

In [ ]:
# Step 12.1: Perform refined KMeans clustering using spatial + environmental features

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Select clustering features
cluster_features = ['latitude', 'longitude', 'temperature', 'humidity', 'pm25']
clustering_data = df[cluster_features].dropna()

# Apply KMeans with 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42)
df.loc[clustering_data.index, 'RefinedCluster'] = kmeans.fit_predict(clustering_data)

# Visualize clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='longitude', y='latitude', hue='RefinedCluster', palette='Set2')
plt.title("Step 12.1: Refined Clustering of Air Quality Regions")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Step 12.2: Perform regression to predict PM2.5 using weather + location + PM10

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Define features and target
features = ['temperature', 'humidity', 'latitude', 'longitude', 'pm10']
target = 'pm25'
regression_data = df[features + [target]].dropna()

# Train-test split
train = regression_data.sample(frac=0.8, random_state=1)
test = regression_data.drop(train.index)

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Step 12.2: Refined Regression Results")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

In [ ]:
!pip install geohash2 folium
# Step 13.1: Bar chart of average PM2.5 across Refined Clusters

import matplotlib.pyplot as plt
import seaborn as sns

cluster_avg = df.groupby('RefinedCluster')['pm25'].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(data=cluster_avg, x='RefinedCluster', y='pm25', palette='viridis')
plt.title('Step 13.1: Average PM2.5 by Refined Cluster')
plt.xlabel('Cluster')
plt.ylabel('Average PM2.5')
plt.tight_layout()
plt.show()

In [ ]:
# Step 13.2: Correlation matrix of numeric columns

numeric_df = df.select_dtypes(include='number').dropna()

plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Step 13.2: Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# Step 13.3: Interactive Folium Map Showing PM2.5 by Geohash

In [ ]:
# Part A: Install geohash2 and folium (only once)
!pip install geohash2 folium

import geohash2

# Generate geohash_5 if it doesn't exist
if 'geohash_5' not in df.columns:
    df['geohash_5'] = df.apply(lambda row: geohash2.encode(row['latitude'], row['longitude'], precision=5), axis=1)

# Decode geohash_5 to latitude and longitude
decoded = df['geohash_5'].apply(geohash2.decode)
df['lat'] = decoded.apply(lambda x: float(x[0]))
df['lon'] = decoded.apply(lambda x: float(x[1]))

In [ ]:
# Part B: Normalize PM2.5 values and create color mapping function

import matplotlib.pyplot as plt
import matplotlib.colors as colors

norm = colors.Normalize(vmin=df['pm25'].min(), vmax=df['pm25'].max())
colormap = plt.colormaps['YlOrRd']

def get_color(value):
    rgba = colormap(norm(value))
    return colors.to_hex(rgba)

In [ ]:
# Part C: Create Folium map and add clustered PM2.5 markers

import folium
from folium.plugins import MarkerCluster

# Center the map on the data
map_center = [df['lat'].mean(), df['lon'].mean()]
m = folium.Map(location=map_center, zoom_start=8)

# Group markers with clustering
marker_cluster = MarkerCluster().add_to(m)

# Add circle markers to the cluster
for _, row in df.iterrows():
    folium.CircleMarker(
        location=(row['lat'], row['lon']),
        radius=6,
        fill=True,
        fill_opacity=0.8,
        color=get_color(row['pm25']),
        fill_color=get_color(row['pm25']),
        popup=folium.Popup(html=f"""
            <b>Geohash:</b> {row['geohash_5']}<br>
            <b>PM2.5:</b> {row['pm25']:.2f}<br>
            <b>Temp:</b> {row['temperature']:.1f}°C<br>
            <b>Humidity:</b> {row['humidity']:.1f}%
        """, max_width=250)
    ).add_to(marker_cluster)

In [ ]:
# Part D: Add a floating legend and display the final map

from branca.element import MacroElement
from jinja2 import Template
import IPython.display as display

# HTML for the PM2.5 scale legend
legend_html = """
{% macro html(this, kwargs) %}
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    width: 170px;
    height: 180px;
    z-index:9999;
    font-size:14px;
    background-color: white;
    padding: 10px;
    border: 2px solid gray;
    border-radius: 5px;">
    <b>PM2.5 Scale</b><br>
    <i style="background:#ffffb2;width:18px;height:18px;display:inline-block"></i> Low<br>
    <i style="background:#fecc5c;width:18px;height:18px;display:inline-block"></i> Moderate<br>
    <i style="background:#fd8d3c;width:18px;height:18px;display:inline-block"></i> High<br>
    <i style="background:#f03b20;width:18px;height:18px;display:inline-block"></i> Very High<br>
    <i style="background:#bd0026;width:18px;height:18px;display:inline-block"></i> Hazardous
</div>
{% endmacro %}
"""

class Legend(MacroElement):
    def __init__(self):
        super().__init__()
        self._template = Template(legend_html)

# Add legend to map
m.get_root().add_child(Legend())

# Display the map
display.display(m)